In [ ]:
import pandas as pd  # Import library pandas untuk manipulasi data

In [ ]:
df = pd.read_csv('../dataset/food_prediction_data.csv', parse_dates=['date'])  # Baca dataset mentah dengan parse tanggal

In [ ]:
# Hapus kolom harga tahun-spesifik dan simpan hanya kolom harga tunggal
year_price_cols = [col for col in df.columns if col.endswith('_2020') or col.endswith('_2024') or col.endswith('_2025')]
if year_price_cols:
    df = df.drop(columns=year_price_cols)
    print(f"Dropped year-specific price columns: {year_price_cols}")

print(df.columns.tolist())  # Tampilkan daftar kolom
df.info()  # Tampilkan info dataset
df.head()  # Tampilkan 5 baris pertama

['date', 'inflation', 'rice_price_2025', 'meat_price_2025', 'chili_price_2025', 'egg_price_2025', 'rice_price_2024', 'meat_price_2024', 'chili_price_2024', 'egg_price_2024', 'rice_price_2020', 'chili_price_2020', 'meat_price_2020', 'egg_price_2020', 'rice_price', 'chili_price', 'meat_price', 'egg_price', 'restaurant_inflation', 'rainfall', 'lag_inflation', 'lag_1', 'lag_2', 'rolling_3']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   date                  71 non-null     datetime64[ns]
 1   inflation             71 non-null     float64       
 2   rice_price_2025       9 non-null      float64       
 3   meat_price_2025       9 non-null      float64       
 4   chili_price_2025      9 non-null      float64       
 5   egg_price_2025        9 non-null      float64       
 6   rice_price_2024       21 non-null     

,date,inflation,rice_price_2025,meat_price_2025,chili_price_2025,egg_price_2025,rice_price_2024,meat_price_2024,chili_price_2024,egg_price_2024,...,rice_price,chili_price,meat_price,egg_price,restaurant_inflation,rainfall,lag_inflation,lag_1,lag_2,rolling_3
0,2020-02-01,0.784603,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,24750.0,32400.0,58150.0,11850.0,NaN,226.586744,NaN,NaN,NaN,NaN
1,2020-03-01,-0.343512,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,25550.0,33100.0,38300.0,11850.0,NaN,188.057164,0.784603,0.784603,NaN,NaN
2,2020-04-01,-0.122868,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,26100.0,31050.0,43100.0,11900.0,NaN,124.365991,-0.343512,-0.343512,0.784603,0.106075
3,2020-05-01,0.049437,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,27750.0,32950.0,45300.0,12350.0,NaN,80.548388,-0.122868,-0.122868,-0.343512,-0.138981
4,2020-06-01,0.558618,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,32050.0,38450.0,36000.0,11500.0,NaN,57.804384,0.049437,0.049437,-0.122868,0.161729


In [ ]:
df["date"] = pd.to_datetime(df["date"])  # Pastikan kolom date dalam format datetime
df = df.sort_values("date")  # Urutkan berdasarkan tanggal
df = df.set_index("date")  # Set date sebagai index

In [ ]:
missing = df.isnull().sum()  # Hitung jumlah nilai hilang per kolom
print(missing)  # Tampilkan jumlah missing values

inflation                0
rice_price_2025         62
meat_price_2025         62
chili_price_2025        62
egg_price_2025          62
rice_price_2024         50
meat_price_2024         58
chili_price_2024        58
egg_price_2024          58
rice_price_2020         23
chili_price_2020        23
meat_price_2020         23
egg_price_2020          23
rice_price               3
chili_price              3
meat_price               3
egg_price                3
restaurant_inflation    59
rainfall                 0
lag_inflation            1
lag_1                    1
lag_2                    2
rolling_3                2
dtype: int64


In [ ]:
# STEP 2 — TANGANI NILAI YANG HILANG
print("=== STEP 2: HANDLING MISSING VALUES ===")

price_cols = [col for col in df.columns if col.endswith("_price") or col.startswith("rice_price") or col.startswith("chili_price") or col.startswith("meat_price") or col.startswith("egg_price")]  # Identifikasi kolom harga

# Interpolasi linear untuk semua kolom harga
if price_cols:
    df[price_cols] = df[price_cols].interpolate(method="linear", limit_direction="both")  # Isi missing dengan interpolasi

# Forward/backward fill untuk nilai hilang yang tersisa
df = df.ffill().bfill()  # Fill forward dan backward

print(f"✅ Missing values handled")
print(f"Remaining NaN: {df.isnull().sum().sum()}")  # Tampilkan sisa NaN

=== STEP 2: HANDLING MISSING VALUES ===
✅ Missing values handled
Remaining NaN: 0


In [ ]:
print(df.describe())  # Tampilkan statistik deskriptif
print(df.head())  # Tampilkan 5 baris pertama
print(df.tail())  # Tampilkan 5 baris terakhir

       inflation  rice_price_2025  meat_price_2025  chili_price_2025  \
count  71.000000        71.000000        71.000000         71.000000   
mean    0.337648     30001.408451     49127.464789      69102.816901   
std     0.824354       417.815051      2085.302115       6269.927587   
min    -1.891279     29900.000000     41600.000000      40250.000000   
25%    -0.260642     29900.000000     49450.000000      71000.000000   
50%     0.270508     29900.000000     49450.000000      71000.000000   
75%     0.889071     29900.000000     49450.000000      71000.000000   
max     1.932848     32650.000000     57950.000000      72400.000000   

       egg_price_2025  rice_price_2024  meat_price_2024  chili_price_2024  \
count       71.000000        71.000000        71.000000         71.000000   
mean     21000.704225     19485.915493     38268.309859      44639.436620   
std         19.807931       678.242640      1157.992087       1316.323763   
min      20900.000000     19100.000000     

In [ ]:
# STEP 3 — FEATURE ENGINEERING
print("=== STEP 3: FEATURE ENGINEERING ===")

# Reset index untuk bekerja dengan date sebagai kolom
df_eng = df.reset_index()

# 1. MOMENTUM INFLASI (rate of change)
df_eng["inflation_momentum"] = df_eng["inflation"].diff(1)  # Hitung perubahan inflasi
print("Added inflation_momentum (inflation rate of change)")

# 2. PERUBAHAN HARGA (% growth)
price_cols = ["rice_price", "chili_price", "meat_price", "egg_price"]  # Kolom harga
for col in price_cols:
    df_eng[f"{col}_pct_change"] = df_eng[col].pct_change() * 100  # Hitung persentase perubahan
print(f"Added price change features for {price_cols}")

# 3. ANOMALI CURAH HUJAN (z-score normalization)
rainfall_mean = df_eng["rainfall"].mean()  # Rata-rata curah hujan
rainfall_std = df_eng["rainfall"].std()  # Standar deviasi curah hujan
df_eng["rainfall_anomaly"] = (df_eng["rainfall"] - rainfall_mean) / rainfall_std  # Hitung anomali
print("Added rainfall_anomaly (z-score normalized)")

# 4. ROLLING MEANS (window 3 & 6)
df_eng["inflation_rolling_3"] = df_eng["inflation"].rolling(window=3, center=False).mean()  # Rolling mean 3 bulan
df_eng["inflation_rolling_6"] = df_eng["inflation"].rolling(window=6, center=False).mean()  # Rolling mean 6 bulan
print("Added rolling means (window 3 & 6)")

# 5. Fill NaN values (baris 1-2 akan NaN dari lag/rolling)
df_eng = df_eng.ffill().bfill()  # Fill missing values
print("Filled NaN values")

print(f"\nDataset shape after feature engineering: {df_eng.shape}")
print(f"Final columns: {list(df_eng.columns)}")

=== STEP 3: FEATURE ENGINEERING ===
Added inflation_momentum (inflation rate of change)
Added price change features for ['rice_price', 'chili_price', 'meat_price', 'egg_price']
Added rainfall_anomaly (z-score normalized)
Added rolling means (window 3 & 6)
Filled NaN values

Dataset shape after feature engineering: (71, 32)
Final columns: ['date', 'inflation', 'rice_price_2025', 'meat_price_2025', 'chili_price_2025', 'egg_price_2025', 'rice_price_2024', 'meat_price_2024', 'chili_price_2024', 'egg_price_2024', 'rice_price_2020', 'chili_price_2020', 'meat_price_2020', 'egg_price_2020', 'rice_price', 'chili_price', 'meat_price', 'egg_price', 'restaurant_inflation', 'rainfall', 'lag_inflation', 'lag_1', 'lag_2', 'rolling_3', 'inflation_momentum', 'rice_price_pct_change', 'chili_price_pct_change', 'meat_price_pct_change', 'egg_price_pct_change', 'rainfall_anomaly', 'inflation_rolling_3', 'inflation_rolling_6']


In [182]:
# STEP 4 — STANDARDIZATION (SCALING)
print("\n=== STEP 4: STANDARDIZATION ===")

# Manual z-score standardization (fit on train only to prevent leakage)
def standardize_column(series):
    """Z-score standardization"""
    return (series - series.mean()) / series.std()

# Identify numeric columns to scale (exclude date and target's lag)
scale_cols = [col for col in df_eng.columns 
              if col not in ['date', 'inflation', 'lag_inflation'] 
              and df_eng[col].dtype in ['float64', 'int64']]

print(f"Scaling {len(scale_cols)} features: {scale_cols[:5]}... (showing first 5)")

# Create a copy for scaled data
df_scaled = df_eng.copy()

# Scale numeric columns
for col in scale_cols:
    if col in df_scaled.columns and df_scaled[col].notna().any():
        df_scaled[col] = standardize_column(df_scaled[col])

print("Features standardized (z-score)")
print(f"Scaled dataset shape: {df_scaled.shape}")


=== STEP 4: STANDARDIZATION ===
Scaling 29 features: ['rice_price_2025', 'meat_price_2025', 'chili_price_2025', 'egg_price_2025', 'rice_price_2024']... (showing first 5)
Features standardized (z-score)
Scaled dataset shape: (71, 32)


In [ ]:
# STEP 5 — TIME SERIES SPLIT (NO SHUFFLE)
print("\n=== STEP 5: TIME SERIES SPLIT ===")

# Define split point - last 3 months for testing
split_date = "2025-10-01"
print(f"Split point: {split_date}")

# Time series split (temporal, no shuffle)
train_df = df_scaled[df_scaled['date'] < split_date].copy()
test_df = df_scaled[df_scaled['date'] >= split_date].copy()

print(f"\nTrain set: {len(train_df)} samples ({train_df['date'].min()} to {train_df['date'].max()})")
print(f" Test set: {len(test_df)} samples ({test_df['date'].min()} to {test_df['date'].max()})")

# Separate features and target
X_train = train_df.drop(['date', 'inflation'], axis=1)
y_train = train_df['inflation']
X_test = test_df.drop(['date', 'inflation'], axis=1)
y_test = test_df['inflation']

print(f"\nFinal dataset dimensions:")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test: {X_test.shape} | y_test: {y_test.shape}")

print(f"\nFeatures ready for modeling:")
print(f"Number of features: {X_train.shape[1]}")
print(f"Features: {list(X_train.columns)[:5]}... (showing first 5)")

print("\nDATA PIPELINE COMPLETE - READY FOR MODELING!")  # Pipeline data selesai


=== STEP 5: TIME SERIES SPLIT ===
Split point: 2025-10-01

Train set: 68 samples (2020-02-01 00:00:00 to 2025-09-01 00:00:00)
 Test set: 3 samples (2025-10-01 00:00:00 to 2025-12-01 00:00:00)

Final dataset dimensions:
X_train: (68, 30) | y_train: (68,)
X_test: (3, 30) | y_test: (3,)

Features ready for modeling:
Number of features: 30
Features: ['rice_price_2025', 'meat_price_2025', 'chili_price_2025', 'egg_price_2025', 'rice_price_2024']... (showing first 5)

DATA PIPELINE COMPLETE - READY FOR MODELING!


In [ ]:
df_scaled.to_csv('../dataset/final_dataset.csv', index=False)  # Simpan dataset final ke CSV